# DCGAN para ArtBench-10

Notebook organizado para:
- usar o **starter pack do professor**;
- carregar o **subset de 20%** do `training_20_percent.csv`;
- treinar um **DCGAN** para imagens RGB 32×32;
- visualizar imagens reais e geradas;
- guardar os pesos do gerador e do discriminador.

O enunciado pede comparação entre famílias de modelos e desenvolvimento inicial no subset de 20%, seguido de treino final no conjunto completo e avaliação com FID/KID. Este notebook fica focado em deixar o **GAN a treinar corretamente primeiro**. 

## 1. Imports

In [1]:
from __future__ import annotations

import sys
import csv
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid

## 2. Configuração

In [3]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 0          # em notebook costuma ser mais estável deixar 0
LATENT_DIM = 100
NUM_EPOCHS = 10
LR = 2e-4
BETA1 = 0.5
SAMPLE_EVERY = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 3. Ajustar caminhos

### Estrutura esperada
Este notebook assume uma estrutura parecida com esta:

```text
projeto/
│
├── notebooks/
│   └── dcgan_artbench_ready.ipynb
├── scripts/
│   └── artbench_local_dataset.py
├── training_20_percent.csv
└── ../ArtBench-10/
```

Se a tua estrutura for diferente, altera apenas estas variáveis.

In [ ]:
def find_project_root(start: Path, markers: list[str], max_levels: int = 6) -> Path:
    """Sobe na árvore de pastas até encontrar uma pasta que contenha todos os markers."""
    path = start
    for _ in range(max_levels):
        if all((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise RuntimeError(
        f"PROJECT_ROOT não encontrado a partir de {start}.\n"
        "Estrutura esperada: pasta com 'ArtBench-10/' e 'TP1-alunos-src-only/'."
    )

PROJECT_ROOT      = find_project_root(Path(".").resolve(), ["ArtBench-10", "TP1-alunos-src-only"])
SCRIPTS_DIR       = PROJECT_ROOT / "TP1-alunos-src-only" / "scripts"
KAGGLE_ROOT       = PROJECT_ROOT / "ArtBench-10"
TRAINING_CSV_PATH = PROJECT_ROOT / "TP1-alunos-src-only" / "student_start_pack" / "training_20_percent.csv"

print("PROJECT_ROOT   =", PROJECT_ROOT)
print("SCRIPTS_DIR    =", SCRIPTS_DIR)
print("KAGGLE_ROOT    =", KAGGLE_ROOT)
print("TRAINING CSV   =", TRAINING_CSV_PATH)

assert SCRIPTS_DIR.exists(), f"Não existe: {SCRIPTS_DIR}"
assert KAGGLE_ROOT.exists(), f"Não existe: {KAGGLE_ROOT}"
assert TRAINING_CSV_PATH.exists(), f"Não existe: {TRAINING_CSV_PATH}"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))


PROJECT_ROOT   = D:\MESTRADO\1º Ano\2º Semestre\IAG\IAG_TP1
SCRIPTS_DIR    = D:\MESTRADO\1º Ano\2º Semestre\IAG\IAG_TP1\TP1-alunos-src-only\scripts
KAGGLE_ROOT    = D:\MESTRADO\1º Ano\2º Semestre\IAG\IAG_TP1\ArtBench-10
TRAINING CSV   = D:\MESTRADO\1º Ano\2º Semestre\IAG\IAG_TP1\TP1-alunos-src-only\student_start_pack\training_20_percent.csv


## 4. Carregar dataset com o helper do professor

In [ ]:
from artbench_local_dataset import load_kaggle_artbench10_splits

hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]

print("Número total de imagens de treino:", len(train_hf))
print("Colunas:", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)

print("Número de classes:", num_classes)
print("Classes:", class_names)

## 5. Transform e dataset PyTorch

In [ ]:
# Para GAN com saída Tanh(), as imagens reais devem ficar em [-1, 1]
transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform is not None else img
        return x, y, real_idx

def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

## 6. Ler o subset de 20% do CSV

In [ ]:
INDEX_COLUMN = "train_id_original"

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    ids = []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if index_column not in (reader.fieldnames or []):
            raise ValueError(f"Coluna {index_column!r} não encontrada. Disponíveis: {reader.fieldnames}")
        for row in reader:
            value = str(row.get(index_column, "")).strip()
            if value:
                ids.append(int(value))
    if len(ids) == 0:
        raise ValueError("Não foram lidos IDs do CSV.")
    return ids

train_ids_from_csv = load_ids_from_training_csv(TRAINING_CSV_PATH, INDEX_COLUMN)

print("Número de IDs lidos do CSV:", len(train_ids_from_csv))
print("Primeiros 10 IDs:", train_ids_from_csv[:10])

## 7. DataLoader do subset

In [ ]:
train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_ids_from_csv)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Tamanho do subset:", len(train_ds))
print("Número de batches:", len(train_loader))

## 8. Ver algumas imagens reais do subset

In [ ]:
x, y, idx = next(iter(train_loader))

grid = make_grid(denorm(x[:36]), nrow=6, padding=2)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Imagens reais do subset de 20%")
plt.show()

print("Exemplos de labels:")
for i in range(min(12, len(y))):
    print(f"{i}: classe={class_names[int(y[i])]} | real_idx={int(idx[i])}")

## 9. Definir o DCGAN

In [ ]:
class Generator32(nn.Module):
    def __init__(self, latent_dim=100, base_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base_ch * 4, 4, 1, 0, bias=False),   # 1 -> 4
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1, bias=False),  # 4 -> 8
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1, bias=False),      # 8 -> 16
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch, 3, 4, 2, 1, bias=False),                 # 16 -> 32
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator32(nn.Module):
    def __init__(self, base_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, base_ch, 4, 2, 1, bias=False),               # 32 -> 16
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_ch, base_ch * 2, 4, 2, 1, bias=False),    # 16 -> 8
            nn.BatchNorm2d(base_ch * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_ch * 2, base_ch * 4, 4, 2, 1, bias=False),# 8 -> 4
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_ch * 4, 1, 4, 1, 0, bias=False),          # 4 -> 1
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)


def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

## 10. Inicializar modelos

In [ ]:
G = Generator32(latent_dim=LATENT_DIM).to(device)
D = Discriminator32().to(device)

G.apply(weights_init)
D.apply(weights_init)

criterion = nn.BCELoss()
optimizer_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

print("Generator params:", sum(p.numel() for p in G.parameters()))
print("Discriminator params:", sum(p.numel() for p in D.parameters()))

## 11. Teste rápido de shapes

In [ ]:
with torch.no_grad():
    z = torch.randn(8, LATENT_DIM, 1, 1, device=device)
    fake = G(z)
    out = D(fake)

print("Fake batch shape:", fake.shape)
print("Discriminator output shape:", out.shape)

## 12. Treino do GAN

In [ ]:
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

g_losses = []
d_losses = []

for epoch in range(NUM_EPOCHS):
    G.train()
    D.train()

    running_g = 0.0
    running_d = 0.0

    for real_images, _, _ in train_loader:
        real_images = real_images.to(device)
        bsz = real_images.size(0)

        real_labels = torch.ones(bsz, device=device)
        fake_labels = torch.zeros(bsz, device=device)

        # ---- train D ----
        optimizer_D.zero_grad()

        out_real = D(real_images)
        loss_real = criterion(out_real, real_labels)

        noise = torch.randn(bsz, LATENT_DIM, 1, 1, device=device)
        fake_images = G(noise)

        out_fake = D(fake_images.detach())
        loss_fake = criterion(out_fake, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # ---- train G ----
        optimizer_G.zero_grad()

        noise = torch.randn(bsz, LATENT_DIM, 1, 1, device=device)
        fake_images = G(noise)
        out_fake_for_g = D(fake_images)

        loss_G = criterion(out_fake_for_g, real_labels)
        loss_G.backward()
        optimizer_G.step()

        running_d += loss_D.item()
        running_g += loss_G.item()

    avg_d = running_d / len(train_loader)
    avg_g = running_g / len(train_loader)
    d_losses.append(avg_d)
    g_losses.append(avg_g)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | D loss = {avg_d:.4f} | G loss = {avg_g:.4f}")

    if (epoch + 1) % SAMPLE_EVERY == 0 or epoch == 0:
        G.eval()
        with torch.no_grad():
            samples = denorm(G(fixed_noise).cpu())

        grid = make_grid(samples[:36], nrow=6, padding=2)
        plt.figure(figsize=(8, 8))
        plt.imshow(grid.permute(1, 2, 0))
        plt.axis("off")
        plt.title(f"Amostras geradas - epoch {epoch+1}")
        plt.show()

## 13. Curvas de loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(g_losses, label="Generator")
plt.plot(d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Curvas de treino do DCGAN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 14. Gerar amostras finais

In [ ]:
G.eval()
with torch.no_grad():
    noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
    fake_images = denorm(G(noise).cpu())

grid = make_grid(fake_images, nrow=8, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Amostras finais geradas")
plt.show()

## 15. Guardar pesos

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(G.state_dict(), OUTPUT_DIR / "dcgan_generator_subset20.pth")
torch.save(D.state_dict(), OUTPUT_DIR / "dcgan_discriminator_subset20.pth")

print("Pesos guardados em:", OUTPUT_DIR.resolve())

## 16. Próximo passo

Quando este notebook estiver a correr bem:
1. guardar algumas amostras para análise qualitativa;
2. criar notebook de avaliação;
3. gerar 5000 imagens;
4. calcular FID/KID;
5. repetir por várias seeds;
6. depois treinar a melhor configuração no conjunto completo.